# CS 3892 / 5892 — Session 6 · First-Order Logic and the Duality

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/cs3892-examples/blob/main/notebooks/cs3892-2026-09-15-first-order-logic-and-duality.ipynb)

**Tuesday, September 15, 2026.** Six examples about quantifiers, in both
SMT-LIB and Python, runnable here with nothing installed.

This notebook does **not** contain copies of the code. It runs the real files in
[`sessions/cs3892-2026-09-15-first-order-logic-and-duality/`](https://github.com/ttj/cs3892-examples/tree/main/sessions/cs3892-2026-09-15-first-order-logic-and-duality),
so what you read here is what CI runs and what the slides showed.

Three ideas run through all six:

1. **A quantifier is not a loop.** Z3 proves things about infinite domains without visiting them.
2. **`sat` is information, not failure.** When you expected a proof and got a model, the model is the answer.
3. **Order and direction are the specification.** Two of these pairs differ only in the order of two quantifiers, or the direction of one arrow.

## Setup

Run this once. On Colab it clones the repo and installs Z3 (a few seconds); anywhere the repo already exists it is a no-op.

In [ ]:
# --- Setup: find the repo (clone on Colab), install Z3, define helpers -------
import os, subprocess, sys, pathlib

REPO_URL = "https://github.com/ttj/cs3892-examples.git"
SESSION  = "cs3892-2026-09-15-first-order-logic-and-duality"

def _find_repo():
    """Walk up from the CWD looking for the repo; otherwise clone it."""
    here = pathlib.Path.cwd()
    for p in [here, *here.parents]:
        if (p / "sessions" / SESSION).is_dir():
            return p
    dest = pathlib.Path("/content/cs3892-examples") if pathlib.Path("/content").is_dir() \
           else pathlib.Path.cwd() / "cs3892-examples"
    if not (dest / "sessions" / SESSION).is_dir():
        print(f"$ git clone {REPO_URL} {dest}")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
    return dest

ROOT = _find_repo()
os.chdir(ROOT)
print("repo:", ROOT)

try:
    import z3
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "z3-solver"], check=True)
    import z3
print("Z3", z3.get_version_string())

SM = ROOT / "sessions" / SESSION / "smt2"
PYD = ROOT / "sessions" / SESSION / "python"

def show(path):
    """Print a source file, so you can read what you are about to run."""
    print(f"--- {pathlib.Path(path).name} " + "-" * max(0, 60 - len(pathlib.Path(path).name)))
    print(pathlib.Path(path).read_text().rstrip())
    print()

def run(path, show_source=True):
    """Run one example and stream its output. Raises if it does not pass.

    .smt2 goes through scripts/run_smt2.py, which checks the file's own
    `; EXPECT:` contract. .py is executed directly and asserts internally.
    The pip wheel for Z3 ships no `z3` CLI, which is why .smt2 is run through
    the Python bindings rather than a shell command -- identical everywhere.
    """
    path = pathlib.Path(path)
    if show_source:
        show(path)
    cmd = ([sys.executable, "scripts/run_smt2.py", str(path)] if path.suffix == ".smt2"
           else [sys.executable, str(path)])
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout.rstrip())
    if r.stderr.strip():
        print(r.stderr.rstrip(), file=sys.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"{path} failed")
    return r.stdout

print("ready — helpers: show(path), run(path)")

## 1. A policy that covers users you have never seen

Slide 21. One assertion binds **every** user. `bob` is declared *after* the
rule and the rule still constrains him — which the propositional encoding on
slide 15 could not do, because it had to name every user in advance.

`unsat` = no such user can exist = the policy is enforced.

In [ ]:
run(SM  / "01_forall_policy.smt2")
run(PYD / "01_forall_policy.py")

## 2. The same policy, with 18 changed to 6

Slide 21. Now it comes back **`sat`** — and that is not the solver failing.
The rule only ever said what happens at twelve months or more. It said nothing
about six, so a six-month user who is *not* approved is entirely consistent
with the policy as written.

The solver just found a hole in your specification. Read the model.

In [ ]:
run(SM  / "02_forall_policy_gap.smt2")
run(PYD / "02_forall_policy_gap.py")

## 3–4. Order matters

Slide 18. These two say different things:

- `ForAll u. Exists s. Sess(u, s)` — everyone has a session, possibly their own
- `Exists s. ForAll u. Sess(u, s)` — **one** session serves everyone

The second implies the first (`03`, `unsat` — no counterexample exists). The
first never implies the second (`04`, `sat` — here is a world where it fails).

In a security property, the difference between the two is usually the
vulnerability.

In [ ]:
run(SM  / "03_order_strong_implies_weak.smt2")
run(PYD / "03_order_strong_implies_weak.py")

In [ ]:
run(SM  / "04_order_weak_not_strong.smt2")
run(PYD / "04_order_weak_not_strong.py")

## 5. "Only A may B" is `B ⇒ A`

Slide 20. *"Only users with an active session may read a document."*

```
correct:   ForAll u,d.  Reads(u,d) => Active(u)
reversed:  ForAll u,d.  Active(u)  => Reads(u,d)
```

`sat` proves these are different claims: there is a world satisfying the first
and violating the second. Z3 finds the simplest one — nobody reads anything,
everybody is active.

The reversal is the most common specification bug there is, and a solver will
prove the reversed version for you without ever mentioning that you asked the
wrong question.

In [ ]:
run(SM  / "05_only_a_may_b.smt2")
run(PYD / "05_only_a_may_b.py")

## 6. Where every counterexample comes from

Slide 19. `¬∀x P(x)  ≡  ∃x ¬P(x)` — De Morgan, one level up, since a `∀` is a
conjunction over the domain and an `∃` is a disjunction. Asserting that the two
sides *differ* is `unsat`, which is the machine-checked version of the proof
done on the board on September 10.

This equivalence **is** the verification loop. "For every reachable state,
nothing bad" negates into "there exists a reachable state where something bad",
and that is the query the solver is actually handed.

In [ ]:
run(SM  / "06_negation_of_forall.smt2")
run(PYD / "06_negation_of_forall.py")

## 7. Encode a problem, ask for a witness

Slide 32. Five regions in a ring, three colours. The formula says nothing about
*how* to colour it — only what a legal colouring **is**. The solver finds one.

This is the shape of most of HW1: describe the legal states, then ask.

In [ ]:
run(SM  / "07_coloring_sat.smt2")
run(PYD / "07_coloring_sat.py")

## 8. `unsat` as a proof of impossibility

Slide 32. Four regions, every one adjacent to every other, three colours.

`unsat` here is not "I looked and did not find one". It is a **proof** that no
legal colouring exists. In today's language: **⟦legal colouring⟧ = ∅** — and
that is exactly why `unsat` is the answer you want when you are trying to show
something *cannot* happen.

In [ ]:
run(SM  / "08_coloring_unsat.smt2")
run(PYD / "08_coloring_unsat.py")

## 9. Now break them

Each of these is a one-line edit in the scratch cell below.

1. In `01`, change `18` to `12`. Still `unsat`? Now try `11`.
2. Add a second rule — *contractors are never approved* — and a user who is both a 20-month employee and a contractor. What happens, and is that what you wanted?
3. In `04`, redefine `Sess(u, s)` as `s == 0` (one shared session). Which of the two orderings now holds?
4. Write *"every document has an owner"* and *"there is one owner for all documents"* and check that one implies the other in exactly one direction.
5. Ask Z3 something undecidable on purpose: quantify over a function and see whether you get `unknown`.
6. In `07`, add a chord across the ring (`r1 != r3`). Still `sat`? Now make it a 5-cycle plus every chord.
7. In `08`, allow a fourth colour. Predict the answer before you run it.

Anything you write here is scratch — the files on disk are untouched.

In [ ]:
from z3 import *

# Scratch. Try one of the exercises above, or anything else.
u = Int("u")
months   = Function("months",   IntSort(), IntSort())
approved = Function("approved", IntSort(), BoolSort())

s = Solver()
s.add(ForAll([u], Implies(months(u) >= 12, approved(u))))

bob = Int("bob")
s.add(months(bob) == 12)        # change this number
s.add(Not(approved(bob)))
print(s.check())
if s.check() == sat:
    print(s.model())

## Where this goes next

| When | What |
|---|---|
| **Thu Sep 17** | SAT — how the search actually works: CNF, DPLL/CDCL, resolution, UNSAT cores |
| **Tue Sep 22** | **Quiz 2** (sets, propositional logic, first-order logic), then SMT proper — DPLL(T), theory solvers, bounded reachability |
| **Thu Sep 24** | **HW1 due** — Z3 from Python, on all of the above |
| **Thu Oct 1** | Project proposal and lightning talk |

Everything here also runs from a terminal:

```bash
git clone https://github.com/ttj/cs3892-examples.git
cd cs3892-examples
pip install z3-solver
bash scripts/check_examples.sh
```

Source: [`sessions/cs3892-2026-09-15-first-order-logic-and-duality/`](https://github.com/ttj/cs3892-examples/tree/main/sessions/cs3892-2026-09-15-first-order-logic-and-duality)